In [ ]:
# Importar librerias
import requests
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from tqdm import tqdm

url = "https://raw.githubusercontent.com/PacktPublishing/Causal-Inference-and-Discovery-in-Python/refs/heads/main/data/hillstrom_clean.csv"

def download_file():
    response = requests.get(url)
    if response.status_code == 200:
        with open("hillstrom_clean.csv", "wb") as f:
            f.write(response.content)
        print("Datos descargados correctamente.")
    else:
        print(f"Error en la descarga. Status code: {response.status_code}")

def read_to_dataframe():
    df = pd.read_csv(url)
    return df

labels_mapping = {0: 'sin_email', 1: 'email_items_mujer', 2: 'email_items_hombre'}
download_file()  # Descargar y guardar los datos
df = read_to_dataframe()  # Cargar los datos en un dataframe

Datos descargados correctamente.


In [ ]:
# Eliminar columnas innecesarias
datos = df.drop(['zip_code__urban', 'channel__web', 'visit', 'conversion'], axis=1)

In [ ]:
# Diccionario con el mapeo de nombres en inglés a español
columnas_espanol = {
    'recency': 'meses_desde_ultima_compra',
    'history': 'historial_de_gasto',
    'mens': 'ha_comprado_items_hombres',
    'womens': 'ha_comprado_items_mujeres',
    'newbie': 'es_nuevo_usuario',
    'spend': 'gasto',
    'zip_code__rural': 'codigo_postal_rural',
    'zip_code__surburban': 'codigo_postal_suburbano',
    'channel__multichannel': 'canal_multicanal',
    'channel__phone': 'canal_telefono',
    'treatment': 'tipo_de_campaña'
}

# Renombrar columnas:
datos = datos.rename(columns=columnas_espanol)

In [ ]:
# Mostramos el número de usuarios:
datos.shape

(64000, 11)

In [ ]:
# Esto es una muestra de nuestros datos:
datos.sample(6)

,meses_desde_ultima_compra,historial_de_gasto,ha_comprado_items_hombres,ha_comprado_items_mujeres,es_nuevo_usuario,gasto,codigo_postal_rural,codigo_postal_suburbano,canal_multicanal,canal_telefono,tipo_de_campaña
11153,3,299.02,0,1,0,0.0,0,0,0,1,1
2349,1,243.67,1,0,0,0.0,0,1,0,0,2
58275,10,351.97,1,0,0,0.0,0,0,0,1,1
430,7,770.87,1,1,1,0.0,0,0,0,1,1
17581,5,34.06,0,1,0,0.0,0,1,0,0,2
46080,9,191.67,0,1,0,0.0,0,0,0,1,0



Variables de entrada:
* **`meses_desde_ultima_compra`**: Indica cuántos meses han pasado desde que el cliente realizó su última compra.

* **`historial_de_gasto`**: Representa el historial acumulado de gastos del cliente a lo largo del año.

* **`ha_comprado_items_hombres`**: Es una variable binaria que indica si el cliente ha realizado compras en la categoría de productos para hombres.

* **`ha_comprado_items_mujeres`**: Similar a la anterior, esta variable binaria indica si el cliente ha comprado productos de la categoría de mujeres.

* **`es_nuevo_usuario`**: Indica si el cliente es nuevo de este año en la plataforma.

* **`codigo_postal_rural`**: Variable binaria que indica si el cliente vive en una zona rural según su código postal.

* **`codigo_postal_suburbano`**: Señala si el cliente reside en una zona suburbana.

* **`canal_multicanal`**: Indica si el cliente utiliza múltiples canales para realizar sus compras.
* **`canal_telefono`**: Muestra si el cliente utiliza el canal telefónico para sus compras o interacciones.

Variable a predecir:

* **`gasto`**: Representa el monto total de dinero gastado por el cliente en un período específico.

Variable sobre la que podemos intervenir:

* **`tipo_de_campaña`**: Especifica el tipo de campaña de marketing o promoción que se ha aplicado o se aplicará al cliente. Esta variable es crucial para medir la efectividad de diferentes estrategias de marketing. Valores:
  * 0: sin email
  * 1: email para mujeres
  * 2: email para hombres

In [ ]:

datos.groupby('tipo_de_campaña').size()

,0
tipo_de_campaña,
0,21306
1,21387
2,21307


In [ ]:
datos.groupby('tipo_de_campaña').gasto.mean()


,gasto
tipo_de_campaña,
0,0.652789
1,1.077202
2,1.422617


In [ ]:
# Preparar features
features = ['meses_desde_ultima_compra', 'historial_de_gasto',
            'ha_comprado_items_hombres', 'ha_comprado_items_mujeres',
            'es_nuevo_usuario', 'codigo_postal_rural',
            'codigo_postal_suburbano', 'canal_multicanal',
            'canal_telefono']

target_col =  'gasto'
treatment_col = 'tipo_de_campaña'

# Dividir los datos por tratamiento
models = {}
metrics = {}
predictions = {}

# Train tebst split estratificado por tratamiento
X = datos[features + [treatment_col]]
y = datos[target_col]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42,
    stratify=datos[treatment_col])

# Guardar los índices de train/test
train_idx = X_train.index
test_idx = X_test.index

In [ ]:
X_train.head()

,meses_desde_ultima_compra,historial_de_gasto,ha_comprado_items_hombres,ha_comprado_items_mujeres,es_nuevo_usuario,codigo_postal_rural,codigo_postal_suburbano,canal_multicanal,canal_telefono,tipo_de_campaña
23000,2,141.64,1,0,1,0,1,0,1,1
41121,10,373.11,1,0,0,0,0,0,0,1
60424,12,133.65,0,1,1,0,0,0,0,1
976,5,543.90,1,1,1,0,0,0,0,0
7415,4,167.09,0,1,0,1,0,0,0,0


## S-Learner

In [ ]:
s_learner = RandomForestRegressor(n_estimators=100, random_state=42)
s_learner.fit(X_train, y_train)

RandomForestRegressor(random_state=42)

In [ ]:
y_pred = s_learner.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
print(f"RMSE: {np.sqrt(mse):.2f}")

RMSE: 15.19


In [ ]:
muestra = pd.DataFrame(X_test.loc[39918]).T
muestra

,meses_desde_ultima_compra,historial_de_gasto,ha_comprado_items_hombres,ha_comprado_items_mujeres,es_nuevo_usuario,codigo_postal_rural,codigo_postal_suburbano,canal_multicanal,canal_telefono,tipo_de_campaña
39918,6.0,1039.2,1.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0


In [ ]:
muestra_0 = muestra.copy()
muestra_0[treatment_col] = 0

muestra_1 = muestra.copy()
muestra_1[treatment_col] = 1

muestra_2 = muestra.copy()
muestra_2[treatment_col] = 2

print(f"Gasto Predicho Campaña 0: ${s_learner.predict(muestra_0)[0]:.2f}")
print(f"Gasto Predicho Campaña 1: ${s_learner.predict(muestra_1)[0]:.2f}")
print(f"Gasto Predicho Campaña 2: ${s_learner.predict(muestra_2)[0]:.2f}")

Gasto Predicho Campaña 0: $8.19
Gasto Predicho Campaña 1: $7.89
Gasto Predicho Campaña 2: $6.76
